# House icons v1: realistic 3D icons for The CurioKids Club and AInation Studio

* About 150 icons: the most-used kids icons, "no emoji shows this" science items, AInation's most-used icons and tech objects, plus general topics for future episodes.
* Style: **realistic 3D product render** (real materials, soft studio light), one object, clean and friendly. No realistic people or faces: those stay on the Fluent emoji.
* Important icons get **2 versions** and future-topic icons get 1 (about 250 pictures). The background is removed automatically with **rembg** (MIT licence), so you get transparent 512 px PNGs.
* Model: **FLUX.1-schnell** (Apache 2.0: free for commercial use). Output: **icons_v1.zip**.

**Settings (right-hand panel):** Session options: Accelerator **GPU T4 x2**, Internet **On**. Add-ons, Secrets: **HF_TOKEN** ticked for this notebook (the same token as before).

**If you ran the first version and got a numpy / `_slice` error:** click the power button to **stop** the session first. A plain "Restart" keeps the broken numpy; a fresh session starts clean. Then import this notebook again.

**Run it in the background (recommended, about 2 hours):** **Save Version → Save & Run All (Commit) → Save**. You can close the tab. When it finishes, open the version → **Output** → download **icons_v1.zip**. Background runs keep their output.

(Or run it by hand: Start session → Run All, keep the tab open, and **download the zip before stopping**: stopping deletes the files.)

In [ ]:
# 1) Install the libraries (about 2 minutes). Kaggle's own numpy version is kept: a newer numpy breaks the model loader.
#    The red "dependency conflicts" lines about bigframes / gradio / cuml / ydata are harmless (those tools are not used).
import numpy; NP = numpy.__version__; print("keeping numpy", NP)
!pip install -q -U "diffusers>=0.32" "transformers>=4.46" accelerate bitsandbytes sentencepiece "numpy=={NP}"
!pip install -q "rembg>=2.0.50" pymatting onnxruntime "numpy=={NP}"
import importlib.metadata as md; print("numpy now", md.version("numpy"), "| rembg", md.version("rembg"))
assert md.version("numpy") == NP, "numpy changed: Stop the session (power button), start it again and Run All."

In [ ]:
# 2) Check the GPU
import torch
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2 (Session options) and run again."
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GB")
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

In [ ]:
# 3) The icon list: (file key, channel, what to draw, background for the cut-out, versions)
#    key = the emoji's code (e.g. 1f4a7 = droplet) so the videos can swap it in; "w_..." = new items with no emoji.
REAL = ("photorealistic 3D product render, highly detailed realistic materials and textures, soft studio lighting, "
        "gentle reflections, crisp focus, clean and friendly, a single object centered with empty space around it, "
        "isolated on a plain solid {bg} background, no text, no letters, no numbers, no logos, no watermark, no people, no hands")
MODEL_STYLE = "clean educational anatomy model, smooth painted plastic, bright friendly colours, "  # body parts: science-class model look
G, P, B = "light grey", "soft pastel pink", "soft pastel blue"   # cut-out backgrounds (white/grey/silver things go on pink)

def k(e):  # emoji -> file key (variation selectors ignored), same as pipeline/icons3d.py
    return "-".join(f"{ord(ch):x}" for ch in e if ord(ch) not in (0xFE0F, 0xFE0E))

ICONS = []
def add(emoji_or_key, ch, what, bg=G, n=2):
    key = emoji_or_key if emoji_or_key.startswith("w_") else k(emoji_or_key)
    ICONS.append((key, emoji_or_key, ch, what, bg, n))

# --- kids: most used (2 versions)
for e, w, bg in [
  ("☀️", "a glowing bright yellow sun with soft rays, like a shiny 3D sphere of light", B),
  ("💧", "a clear glossy water droplet with realistic refraction and highlights", P),
  ("💨", "a swirling gust of wind made of soft white mist trails", B),
  ("🌙", "a golden crescent moon with soft craters", B),
  ("✨", "three sparkling golden four-point stars with glints", B),
  ("🌊", "a curling ocean wave with clear turquoise water and white foam", P),
  ("☁️", "a fluffy soft white cloud", B),
  ("🌍", "planet Earth globe with blue oceans, green land and soft clouds", P),
  ("🌧️", "a grey rain cloud with falling clear raindrops", P),
  ("🌈", "a bright rainbow arc with fluffy clouds at both ends", B),
  ("⚡", "a glossy yellow lightning bolt", B),
  ("❄️", "a delicate white crystal snowflake", B),
  ("🌱", "a small green seedling with two leaves growing from a mound of dark soil", P),
  ("💡", "a glowing glass light bulb with a warm filament", B),
  ("🍎", "a shiny red apple with a green leaf", G),
  ("🌸", "a pink cherry blossom flower", B),
  ("⭐", "a glossy golden five-point star", B),
  ("🧠", MODEL_STYLE + "a pink human brain model", B),
  ("👂", MODEL_STYLE + "a human ear model", B),
  ("🐝", "a cute fuzzy honeybee with translucent wings", B),
  ("🦴", "a clean white bone", P),
  ("🌰", "a brown acorn seed with its cap", B),
  ("🍃", "two fresh green leaves blowing in the wind", P),
  ("🧊", "a clear ice cube with frosty edges", P),
  ("🍂", "an orange and red autumn maple leaf", B),
  ("🌅", "a sun rising over a calm sea horizon, small round scene", P),
  ("🌀", "a small swirling tornado made of soft grey-blue wind", P),
  ("🪐", "the planet Saturn with its rings", P),
  ("🌑", "a dark grey moon with craters", P),
  ("🦋", "a colourful blue butterfly with open wings", P),
  ("🔥", "a bright orange flame", B),
  ("🌳", "a round green leafy tree with a brown trunk", P),
  ("🪨", "a smooth grey stone rock", P),
  ("🩸", "a glossy red drop of blood", B),
  ("🪥", "a toothbrush with blue bristles and a dab of white toothpaste", P),
  ("🧼", "a bar of soap with soap bubbles", B),
  ("🥤", "a paper cup with a lid and a striped straw", B),
  ("🌋", "a small volcano erupting with glowing lava", B),
  ("🌿", "a sprig of fresh green herb leaves", P),
  ("🫧", "a cluster of shiny iridescent soap bubbles", P),
  ("🐶", "a cute puppy dog, full body, sitting", B),
  ("🍯", "a honey pot with a wooden honey dipper", B),
  ("🪶", "a soft white feather", P),
  ("🌕", "a bright full moon with craters", B),
  ("🔍", "a magnifying glass with a wooden handle", P),
  ("🚀", "a white and red toy rocket with a flame", B),
  ("🦷", MODEL_STYLE + "a clean white healthy molar tooth", P),
  ("🦠", "a cute round green microbe germ with little bumps", P),
  ("🫁", MODEL_STYLE + "a pair of human lungs", B),
  ("🧲", "a red and silver horseshoe magnet", B),
  ("🌡️", "a glass thermometer with red liquid", P),
]: add(e, "kids", w, bg)
# --- things no emoji shows well (2 versions)
for key, w, bg in [
  ("w_tooth_cavity", MODEL_STYLE + "a molar tooth with a small dark cavity hole", P),
  ("w_tongue_model", MODEL_STYLE + "a human tongue with visible taste buds", B),
  ("w_inner_ear", MODEL_STYLE + "a cutaway of the inner ear with the eardrum and the snail-shaped cochlea", B),
  ("w_heart_model", MODEL_STYLE + "a human heart", B),
  ("w_stomach_model", MODEL_STYLE + "a human stomach", B),
  ("w_eye_model", MODEL_STYLE + "a human eyeball with a blue iris", P),
  ("w_soil_block", "a cut-away cube of ground showing layers: grass, dark soil with roots, brown clay, sand and stones", B),
  ("w_seed_cut", "a bean seed cut in half showing the tiny baby plant inside", B),
  ("w_plant_cell", "a green plant cell model cut open showing its parts", P),
  ("w_sound_wave", "a speaker sending out glowing blue sound waves", P),
  ("w_prism", "a glass prism splitting a beam of white light into a rainbow", G),
  ("w_skin_layers", MODEL_STYLE + "a small block of human skin layers with a hair", B),
]: add(key, "kids", w, bg)
# --- AInation: most used + tech (2 versions)
for e, w, bg in [
  ("💬", "a glossy white speech bubble", P), ("✅", "a glossy green check mark", P), ("⚠️", "a yellow warning triangle sign with an exclamation mark shape", B),
  ("📄", "a sheet of paper document with grey lines of text", P), ("🛡️", "a glossy blue and silver shield", P), ("🎯", "a red and white dartboard target with a dart in the centre", B),
  ("📖", "an open book with blank pages", B), ("🤖", "a friendly modern white robot with blue glowing eyes, full body", P), ("❌", "a glossy red cross mark", B),
  ("🔒", "a shiny golden padlock, closed", B), ("📱", "a modern smartphone with a blue glowing screen", P), ("📅", "a desk calendar with blank pages", P),
  ("✉️", "a white envelope", P), ("🧩", "a single blue jigsaw puzzle piece", P), ("🪙", "a shiny gold coin", B), ("💼", "a brown leather briefcase", B),
  ("🖼️", "a picture frame with a landscape photo", P), ("🔗", "two linked silver chain links", P), ("📺", "a modern flat TV on a stand", P),
  ("🎛️", "an audio mixer with knobs and sliders", P), ("🚫", "a red prohibition sign", B), ("🎲", "a white dice with black dots", P),
  ("✏️", "a yellow pencil", B), ("📞", "a retro telephone handset", B), ("✂️", "a pair of scissors", P), ("📚", "a stack of three colourful books", B),
  ("🛑", "a red stop sign", B), ("📝", "a notepad with a pencil", P), ("🧹", "a broom", B), ("📰", "a folded newspaper", P),
  ("🔮", "a crystal ball glowing purple", G), ("🗺️", "a folded paper world map", P), ("🎙️", "a studio microphone", P), ("📷", "a modern camera", P),
  ("⚖️", "a golden balance scale", B), ("🩺", "a stethoscope", P), ("🎣", "a fishing hook on a line, shiny metal", P), ("🏷️", "a price tag label with a string", B),
  ("🔑", "a golden key", B), ("💻", "a modern silver laptop with a glowing screen", P), ("⌨️", "a computer keyboard", P), ("🖥️", "a desktop computer monitor", P),
  ("w_ai_chip", "a computer processor chip with glowing blue circuit lines", P), ("w_server", "a modern server rack with blue lights", P),
  ("w_neural_net", "a glowing blue network of connected spheres and lines, 3D", P), ("w_cloud_data", "a white cloud with glowing blue data lines below it", P),
  ("w_password", "a password field with hidden dots and a padlock", P), ("w_deepfake_mask", "a white theatre mask on a digital glowing grid", P),
]: add(e, "ain", w, bg)
# --- general topics for future episodes (1 version)
for e, w, bg in [
  ("🦕", "a cute green long-neck dinosaur", P), ("🐘", "a cute baby elephant", P), ("🦁", "a friendly lion", B), ("🐱", "a cute kitten sitting", B),
  ("🐟", "a colourful tropical fish", P), ("🐋", "a blue whale", P), ("🦈", "a friendly shark", P), ("🐦", "a small blue bird", P),
  ("🦉", "a cute owl", B), ("🐧", "a penguin", P), ("🐸", "a green frog", P), ("🐢", "a sea turtle", P), ("🕷️", "a cute cartoon-proportioned spider", P),
  ("🐜", "an ant", P), ("🐌", "a snail", B), ("🐙", "a friendly orange octopus", B), ("🌻", "a sunflower", B), ("🌵", "a green cactus in a pot", P),
  ("🍄", "a red mushroom with white spots", B), ("🥕", "a carrot with leaves", B), ("🍌", "a banana", B), ("🥛", "a glass of milk", P),
  ("🍞", "a loaf of bread", B), ("🧀", "a wedge of cheese", B), ("🥚", "an egg", P), ("⏰", "a red alarm clock", B), ("🎒", "a school backpack", B),
  ("🎨", "a paint palette with brushes", B), ("🎵", "a glossy music note", B), ("🥁", "a toy drum", B), ("🎸", "an acoustic guitar", B),
  ("⚽", "a football", B), ("🪁", "a colourful kite", B), ("🚲", "a bicycle", P), ("🚗", "a small red car", B), ("🚂", "a toy steam train", B),
  ("✈️", "a white passenger airplane", P), ("⛵", "a sailboat", P), ("🏠", "a small house", B), ("⛰️", "a mountain with a snowy top", P),
  ("☂️", "an open red umbrella", B), ("🔋", "a battery", P), ("🔭", "a telescope on a tripod", P), ("🔬", "a microscope", P),
  ("🧪", "a glass test tube with green liquid", P), ("⚛️", "an atom model with glowing electron orbits", P), ("🧬", "a DNA double helix", P),
  ("🧭", "a compass", B), ("🏆", "a golden trophy cup", B), ("🎁", "a gift box with a ribbon", B), ("🎈", "a red balloon", B),
  ("🩹", "an adhesive bandage", B), ("🧴", "a lotion bottle", B), ("⏳", "an hourglass with sand", P),
]: add(e, "general", w, bg, n=1)

seen = set(); ICONS = [i for i in ICONS if not (i[0] in seen or seen.add(i[0]))]
TOTAL = sum(i[5] for i in ICONS)
print(len(ICONS), "icons,", TOTAL, "pictures")

In [ ]:
# 4) Load FLUX.1-schnell in 4-bit (first time downloads ~33 GB; about 5-10 minutes)
import os, gc
from diffusers import FluxPipeline, FluxTransformer2DModel, BitsAndBytesConfig as DiffBnb
from transformers import T5EncoderModel, BitsAndBytesConfig as TfBnb
MODEL = "black-forest-labs/FLUX.1-schnell"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN"); print("using the HF_TOKEN secret")
except Exception:
    raise SystemExit("No HF_TOKEN secret: Add-ons -> Secrets -> HF_TOKEN (tick it for this notebook), then run again.")
q = dict(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)
transformer = FluxTransformer2DModel.from_pretrained(MODEL, subfolder="transformer", quantization_config=DiffBnb(**q), torch_dtype=DTYPE)
text_encoder_2 = T5EncoderModel.from_pretrained(MODEL, subfolder="text_encoder_2", quantization_config=TfBnb(**q), torch_dtype=DTYPE)
pipe = FluxPipeline.from_pretrained(MODEL, transformer=transformer, text_encoder_2=text_encoder_2, torch_dtype=DTYPE)
pipe.enable_model_cpu_offload(); pipe.vae.enable_tiling()
gc.collect(); torch.cuda.empty_cache(); print("model ready")

In [ ]:
# 5) Background remover (rembg, MIT licence; downloads its small model once)
from rembg import new_session, remove
try:
    RB = new_session("isnet-general-use")
except Exception as e:
    print("isnet not available, using u2net:", e); RB = new_session("u2net")
print("cut-out ready")

In [ ]:
# 6) Make the icons. Already-made files are skipped, so a re-run continues where it stopped.
import json, time, datetime
import numpy as np
from PIL import Image
OUT = "/kaggle/working/icons_v1"; os.makedirs(OUT + "/png", exist_ok=True); os.makedirs(OUT + "/raw", exist_ok=True)
SEEDS = [7, 21]
manifest = []; n = 0; t0 = time.time()
def cutout(img):
    rgba = remove(img, session=RB, post_process_mask=True)
    a = np.asarray(rgba)[:, :, 3]
    ys, xs = np.where(a > 10)
    if len(xs) == 0: return None
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    side = int(max(x1 - x0, y1 - y0) * 1.08) + 2
    sq = Image.new("RGBA", (side, side), (0, 0, 0, 0))
    sq.paste(rgba.crop((x0, y0, x1 + 1, y1 + 1)), ((side - (x1 - x0)) // 2, (side - (y1 - y0)) // 2))
    return sq.resize((512, 512), Image.LANCZOS)
for key, label, ch, what, bg, nver in ICONS:
    prompt = what + ", " + REAL.format(bg=bg)
    for seed in SEEDS[:nver]:
        n += 1
        name = f"{key}_s{seed}"
        png, raw = f"{OUT}/png/{name}.png", f"{OUT}/raw/{name}.jpg"
        if not os.path.exists(png):
            t = time.time()
            img = pipe(prompt, width=768, height=768, num_inference_steps=4, guidance_scale=0.0, max_sequence_length=256,
                       generator=torch.Generator("cpu").manual_seed(seed)).images[0]
            if np.asarray(img).mean() < 3: print("WARNING: black picture for", name)
            img.save(raw, quality=88)
            cut = cutout(img)
            if cut is None: print("WARNING: cut-out found nothing for", name); continue
            cut.save(png, optimize=True)
            print(f"[{n}/{TOTAL}] {label} {name} ({ch}) in {time.time() - t:.0f}s")
        manifest.append({"file": f"png/{name}.png", "key": key, "label": label, "channel": ch, "seed": seed, "prompt": prompt,
                         "model": MODEL, "steps": 4, "made_at": [768, 768], "size": [512, 512], "cutout": "rembg isnet-general-use",
                         "licence": "Apache-2.0 (FLUX.1-schnell); rembg MIT", "date": datetime.date.today().isoformat()})
json.dump(manifest, open(OUT + "/manifest.json", "w"), indent=1, ensure_ascii=False)
print(f"done: {len(manifest)} icons in {(time.time() - t0) / 60:.0f} min")

In [ ]:
# 7) Contact sheets (icons on a dark and on a light background, to check the cut-out edges) + the zip
from PIL import ImageDraw
def sheet(bgcol, fn):
    tiles = []
    for m in manifest:
        im = Image.open(os.path.join(OUT, m["file"])).convert("RGBA"); im.thumbnail((150, 150))
        t = Image.new("RGB", (160, 180), bgcol); t.paste(im, (5, 5), im)
        ImageDraw.Draw(t).text((4, 162), (m["label"] + " " + str(m["seed"]))[:24], fill="white" if bgcol != "white" else "black"); tiles.append(t)
    cols = 12; rows = (len(tiles) + cols - 1) // cols
    s = Image.new("RGB", (cols * 160, rows * 180), bgcol)
    for i, t in enumerate(tiles): s.paste(t, ((i % cols) * 160, (i // cols) * 180))
    s.save(os.path.join(OUT, fn), quality=85); return s
sheet("#1F2D44", "sheet_dark.jpg"); s = sheet("white", "sheet_light.jpg")
import shutil
shutil.make_archive("/kaggle/working/icons_v1", "zip", OUT)
print("zip:", round(os.path.getsize("/kaggle/working/icons_v1.zip") / 2**20, 1), "MB")
print("NOW: download icons_v1.zip from Output (if you ran it by hand: BEFORE stopping the session).")
s